In [1]:
import pandas as pd
import duckdb

from juptils import pretty
from juptils import lefty

# Supplier queries

In [2]:
monthly_sales = pd.read_csv("./csv/monthly-sales.csv")
#monthly_sales

In [49]:
import ipywidgets as widgets
from ipywidgets import interact

def my_query(supplier):

    x = supplier
    supplier_clause = {
        "nippon-metal":"supplier = 'nippon-metal'",
        "us-steel":"supplier = 'us-steel'",
    }.get(x, "1=1")

    sql_template = """SELECT 
            SUBSTR(month, 1, 4) AS year, 
            supplier AS supplier,
            format('{:t,}', SUM(month_amt)) AS ytd_amt
        FROM monthly_sales
        WHERE _SUPPLIER_
        GROUP BY year, supplier
        ORDER BY year, supplier
        """.replace("_SUPPLIER_", supplier_clause)
    
    df = duckdb.query(sql_template).df() 
    df2 = df.style.set_properties(subset=['supplier'],  **{'text-align': 'left'}).set_properties(subset=['ytd_amt'], **{'text-align': 'right'}).hide(axis="index")
    
    display(df2)

#my_query("all")    
interact(my_query, supplier=['all','nippon-metal', 'us-steel']);

interactive(children=(Dropdown(description='supplier', options=('all', 'nippon-metal', 'us-steel'), value='all…

## Yearly Sales

In [41]:
# NOTE: '{:t,}'  works, but '{:t}' doesn't work. for floats: '{:t,.2f}'

df = duckdb.query("""
SELECT 
    SUBSTR(month, 1, 4) AS year, 
    supplier AS supplier, 
    format('{:t,}', SUM(month_amt)) AS ytd_amt

FROM monthly_sales
GROUP BY year, supplier
ORDER BY year, supplier
""").df()

df.style.set_properties(subset=['supplier'],  **{'text-align': 'left'}).set_properties(subset=['ytd_amt'], **{'text-align': 'right'})
    


#.style.set_properties(**{'white-space': 'pre'})
#.lefty() 

#interact(f, x=[('one', 10), ('two', 20)]);

,year,supplier,ytd_amt
0,2023,nippon-metal,"9,360,700"
1,2023,us-steel,"9,675,200"
2,2024,nippon-metal,"9,489,700"
3,2024,us-steel,"8,797,300"
4,2025,nippon-metal,"11,298,600"
5,2025,us-steel,"13,043,700"
6,2026,nippon-metal,"1,153,000"
7,2026,us-steel,"306,300"


# Quarterly Sales

In [4]:
duckdb.query("""
SELECT 
    SUBSTR(month, 1, 4) || '-Q' || 
    CAST(CEIL(CAST(SUBSTR(month, 6, 2) AS INT64) / 3.0) AS INT64) AS quarter,
    supplier,
    SUM(month_amt) AS quarter_total
FROM monthly_sales
GROUP BY quarter, supplier
ORDER BY quarter, supplier
""").df()

,quarter,SUPPLIER,quarter_total
0,2023-Q1,nippon-metal,709400.0
1,2023-Q1,us-steel,569200.0
2,2023-Q2,nippon-metal,2388200.0
3,2023-Q2,us-steel,3766900.0
4,2023-Q3,nippon-metal,3842200.0
5,2023-Q3,us-steel,3007500.0
6,2023-Q4,nippon-metal,2420900.0
7,2023-Q4,us-steel,2331600.0
8,2024-Q1,nippon-metal,3053200.0
9,2024-Q1,us-steel,2446100.0
